# NF1 · Almacenamiento y gestión de datos (RA1)
## La memoria de las partidas — plataforma de juego online

**Tu rol:** Ingeniero/a de Datos del estudio. **Tu misión:** construir la primera
capa del *data lake* del juego: ingerir telemetría cruda de tres fuentes
(sesiones, partidas, matchmaking), limpiarla y almacenarla en Parquet + **Delta Lake**.

> Autoevaluable: la celda final genera `resultados.json`. Sigue las **reglas de
> limpieza al pie de la letra** (hacen tu resultado reproducible).

### Lo que demostrarás (RA1)
1.2 ingesta multiformato · 1.1 diagnóstico de calidad · 1.4 limpieza+features ·
1.3 formato/benchmark · 1.5 comunicación.

### Antes de empezar · genera los datos (una sola vez)

Abre una **terminal** (no una celda) y ejecuta, **desde la raíz del repositorio**:

```bash
cd nf1
python datos/generar_datos.py --salida datos/raw
```

Este cuaderno **se sitúa solo** en `nf1/`, así que todas sus rutas son relativas a esa
carpeta. Si la primera celda de código falla con un error de fichero no encontrado, es
que te falta este paso.


In [ ]:
import os
if os.path.basename(os.getcwd()) == "actividad": os.chdir("..")  # ejecutar desde la carpeta del núcleo (donde está datos/)
import json, os, shutil, time
import numpy as np, pandas as pd, pyarrow as pa
import matplotlib.pyplot as plt
from deltalake import DeltaTable, write_deltalake
RAW, PROC = "datos/raw", "datos/procesado"; os.makedirs(PROC, exist_ok=True)
resultados = {}
print("Entorno listo.")

---
## Fase 1 · Ingesta (RA1.2)
Carga `sesiones.csv`, `partidas.json` (aplánalo con `pd.json_normalize`) y
`matchmaking.parquet`.

In [ ]:
# Fase 1 · Ingesta — completa las 4 cargas
df = ...           # TODO: carga el CSV de sesiones · pista: pd.read_csv sobre datos/raw/sesiones.csv
partidas = ...     # TODO: abre el JSON de partidas · pista: json.load(open(..., encoding="utf-8"))
df_p = ...         # TODO: aplana el JSON anidado a tabla · pista: pd.json_normalize
df_extra = ...     # TODO: carga el parquet de matchmaking · pista: pd.read_parquet
resultados["fase1_ingesta"] = {"filas_csv": int(len(df)), "filas_json": int(len(df_p)),
    "filas_extra": int(len(df_extra)), "columnas_json_aplanado": sorted(df_p.columns.tolist())}
resultados["fase1_ingesta"]

---
## Explora tus datos (antes de limpiar)

Conoce con qué trabajas antes de tocar nada. **Diccionario de las tres fuentes:**

- **`sesiones.csv`** → `df`: telemetría por sesión.
  - `id_jugador`, `timestamp`, **`ping_ms`** = latencia en ms (la *métrica* a limpiar: tiene **nulos y outliers**),
    **`fps`** = fotogramas/seg guardado como **texto** (con el centinela `"sin_dato"`),
    `latitud` / `longitud` = geolocalización aproximada del jugador, la que usa el emparejador
    para asignarle servidor regional (no hay que limpiarlas).
- **`partidas.json`** → `df_p` (ya aplanado): datos de cada partida.
  - `id_partida`, `timestamp`, `ubicacion.lat` / `ubicacion.lon` = zona del servidor que alojó la partida,
    **`datos.mapa`** = categoría con mayúsculas e idiomas mezclados (a estandarizar), `datos.jugadores`.
- **`matchmaking.parquet`** → `df_extra`: `id_jugador`, `timestamp`, **`mmr`** guardado como **texto**.

La celda siguiente te muestra las primeras filas, el **tipo** de cada columna y los nulos.


In [ ]:
# Vistazo a las 3 fuentes antes de trabajar (esta celda NO se evalúa)
print("— sesiones (df) —"); display(df.head()); print(df.dtypes, "\n")
print("nulos por columna (sesiones):", df.isna().sum().to_dict(), "\n")
print("— partidas (df_p) —"); display(df_p.head(3))
print("— matchmaking (df_extra) —"); display(df_extra.head(3))

---
## Fase 2 · Diagnóstico (RA1.1)
Mide, antes de limpiar: nulos en `ping_ms`; outliers de `ping_ms` con la regla de
rango (`<0` o `>500`); nulos de texto en `fps` (marcados como `"sin_dato"`).

In [ ]:
# Fase 2 · Diagnóstico (mide ANTES de limpiar)
nulos_metrica = ...   # TODO: nº de nulos en ping_ms · pista: .isna().sum()
n_outliers = ...      # TODO: nº de valores fuera de rango (<0 o >500) en ping_ms · pista: máscara booleana + .sum()
nulos_texto = ...     # TODO: nº de "sin_dato" en fps · pista: (columna == "sin_dato").sum()
resultados["fase2_diagnostico"] = {"nulos_metrica": int(nulos_metrica),
    "n_outliers_metrica": int(n_outliers), "nulos_temp_texto": int(nulos_texto)}
resultados["fase2_diagnostico"]

---
## Fase 3 · Limpieza (RA1.4) — reglas exactas
1. `fps` → numérico (`"sin_dato"`→NaN) e imputar con la **MEDIANA**.
2. `ping_ms`: outliers de rango (`<0` o `>500`) → NaN; imputar TODO con la **MEDIANA** de los válidos.
3. `datos.mapa`: minúsculas + mapeo canónico `{forest/bosque→bosque, desert/desierto→desierto, snow/nieve→nieve}`.
4. `mmr` (del extra) → numérico.

In [ ]:
# Fase 3 · Limpieza (sigue las reglas exactas)
CAT = {"forest":"bosque","bosque":"bosque","desert":"desierto","desierto":"desierto","snow":"nieve","nieve":"nieve"}
# TODO 1 (fps): pásalo a numérico ("sin_dato"->NaN) e imputa con la MEDIANA
#         pista: pd.to_numeric(..., errors="coerce") y luego .fillna(<mediana>)
# TODO 2a (ping_ms): calcula la MEDIANA usando SOLO los valores en rango [0, 500]
# TODO 2b (ping_ms): marca como NaN los outliers (<0 o >500) e imputa TODOS los nulos con esa mediana
#         pista: máscara booleana para poner NaN y luego .fillna(<mediana del paso 2a>)
# TODO 3 (datos.mapa): quita espacios, pon en minúsculas y mapea con CAT
#         pista: .str.strip().str.lower().map(CAT)
# TODO 4 (mmr): pásalo a numérico · pista: pd.to_numeric(df_extra["mmr"], errors="coerce")
nulos_restantes = int(df[["ping_ms","fps"]].isna().sum().sum())
categorias_finales = sorted(df_p["datos.mapa"].dropna().unique().tolist())
print("Comprobación · nulos restantes por columna:", df[["ping_ms","fps"]].isna().sum().to_dict(), "(deben ser 0)")
resultados["fase3_limpieza"] = {"nulos_restantes_total": nulos_restantes,
    "n_categorias_estandarizadas": len(categorias_finales), "categorias_finales": categorias_finales}
resultados["fase3_limpieza"]

---
## Fase 4 · Feature engineering (RA1.4)
Crea `hora_del_dia` y `dia_semana` a partir de `timestamp`.

In [ ]:
# Fase 4 · Features temporales
cols_antes = set(df.columns)   # foto de las columnas ANTES de crear las nuevas (no lo toques)
# TODO: convierte timestamp a datetime y crea las columnas hora_del_dia y dia_semana
#       pista: pd.to_datetime(df["timestamp"]); luego .dt.hour y .dt.dayofweek

columnas_nuevas = sorted(set(df.columns) - cols_antes)   # se DERIVA de tu df: si no las creas, sale vacía
if not columnas_nuevas:
    print("⚠️  Autochequeo: no has creado hora_del_dia ni dia_semana. Completa el TODO y vuelve a ejecutar esta celda.")
resultados["fase4_features"] = {"columnas_nuevas": columnas_nuevas, "filas_finales": int(len(df))}
resultados["fase4_features"]

---
## Fase 5 · Almacenamiento + benchmark (RA1.3)
Guarda las sesiones limpias en CSV, Parquet y **Delta**, mide tamaños y tiempos, y
dibuja un gráfico de barras de tamaños.

In [ ]:
# Fase 5 · Almacenamiento + benchmark
def tam_mb(p):
    if os.path.isdir(p): return round(sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(p) for f in fs)/1e6,3)
    return round(os.path.getsize(p)/1e6,3)
p_csv,p_pq,p_d = f"{PROC}/sesiones.csv",f"{PROC}/sesiones.parquet",f"{PROC}/sesiones_delta"
# TODO: guarda df en los TRES formatos (esta es tu parte)
#   1) CSV:     df.to_csv(p_csv, index=False)
#   2) Parquet: df["timestamp"] = df["timestamp"].astype(str); df.to_parquet(p_pq, index=False)
#   3) Delta:   write_deltalake(p_d, pa.Table.from_pandas(df, preserve_index=False), mode="overwrite")

# --- Benchmark (ya resuelto): tamaños, tiempos de lectura y gráfico ---
assert os.path.exists(p_d), "Aún no has escrito la tabla Delta (TODO 3 de esta celda)."
delta_version = DeltaTable(p_d).version()
tam_csv, tam_pq, tam_d = tam_mb(p_csv), tam_mb(p_pq), tam_mb(p_d)
t0=time.time(); pd.read_csv(p_csv);    t_csv=round(time.time()-t0,4)
t0=time.time(); pd.read_parquet(p_pq); t_pq =round(time.time()-t0,4)
plt.bar(["CSV","Parquet","Delta"],[tam_csv,tam_pq,tam_d]); plt.ylabel("MB"); plt.title("Tamaño por formato"); plt.show()
resultados["fase5_almacenamiento"] = {"tamano_mb":{"csv":tam_csv,"parquet":tam_pq,"delta":tam_d},
    "tiempo_lectura_s":{"csv":t_csv,"parquet":t_pq},"reduccion_pct_parquet_vs_csv":round(100*(tam_csv-tam_pq)/tam_csv,1),
    "delta_version":int(delta_version),"filas_guardadas":int(len(df)),"registros_mongo":-1}
resultados["fase5_almacenamiento"]

---
## Fase 6 · Razonamiento (formato examen, RA1)
En **máximo 12 líneas**:
**a)** Justifica por qué **Parquet/Delta** para la capa procesada del data lake del
juego frente a CSV (columnar, compresión, coste de escaneo) y qué aporta **Delta**.
**b)** Tienes una API de partidas que devuelve JSON anidado y un panel en vivo que
necesita leer una partida completa con baja latencia: ¿data lake o **NoSQL**?
Distingue una *observación* de una *decisión accionable* para el estudio.

*(Escribe aquí tu respuesta.)*


---
## Celda final · Generar `resultados.json` (no modificar)

In [ ]:
ALUMNO = "TU_NOMBRE_Y_APELLIDOS"   # <-- pon aquí "Apellidos, Nombre"
resultados["metadata"] = {"caso":"gaming","seed":42,"alumno":ALUMNO}
assert ALUMNO != "TU_NOMBRE_Y_APELLIDOS", "⚠️ Pon tus Apellidos, Nombre en ALUMNO antes de entregar."
assert set(resultados) >= {"fase1_ingesta","fase2_diagnostico","fase3_limpieza","fase4_features","fase5_almacenamiento"}, "Faltan fases por completar."
json.dump(resultados, open("resultados.json","w",encoding="utf-8"), ensure_ascii=False, indent=2)
print("✅ resultados.json generado. Revisa que 'alumno' tiene tu nombre. Entrega el .ipynb Y resultados.json.")